In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('train_data.csv')
df = df.dropna(subset=['JobSatLevel'])

C:\Users\eltou\AppData\Local\Temp\ipykernel_3212\4267432579.py:1: DtypeWarning: Columns (0: ResponseId) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('train_data.csv')


In [3]:
sat_mapping = {'Low': 0, 'Average': 1, 'High': 2}

df['JobSat_Score'] = df['JobSatLevel'].map(sat_mapping)

selected_features = [
    'JobSatLevel',  # Keep your target!
    'JobSat_Score', # Keep your encoded target for future math
    'RemoteWork', 
    'Employment', 
    'LearnCodeChoose', 
    'DatabaseHaveWorkedWith', 
    'DatabaseAdmired'
]
df_2 = df[selected_features].copy()

# Null handling

In [4]:
# Professional Null Report
null_report = pd.DataFrame({
    'Missing Values': df_2.isnull().sum(),
    'Percentage (%)': (df_2.isnull().sum() / len(df_2)) * 100
}).sort_values(by='Missing Values', ascending=False)

print(null_report)

                        Missing Values  Percentage (%)
DatabaseAdmired                  11634       48.553900
DatabaseHaveWorkedWith            5974       24.932181
RemoteWork                        3484       14.540295
LearnCodeChoose                     17        0.070949
JobSatLevel                          0        0.000000
JobSat_Score                         0        0.000000
Employment                           0        0.000000


In [5]:
# Learn Code Choose
df_2['LearnCodeChoose'] = df_2['LearnCodeChoose'].fillna('No, I am not new to coding and did not learn new coding techniques or programming languages')
# Remote Work
df_2['RemoteWork'] = df_2['RemoteWork'].fillna('Not Specified')
# DatabaseHaveWorkedWith
df_2['DatabaseAdmired'] = df_2['DatabaseAdmired'].fillna('None')
# DatabaseAdmired
df_2['DatabaseHaveWorkedWith'] = df_2['DatabaseHaveWorkedWith'].fillna('None')

print("Nulls remaining:", df_2.isnull().sum().sum())

Nulls remaining: 0


# Encoding

In [6]:
# Unique Values for each column
# 1. Check the exact strings for standard categorical columns
categorical_cols = ['Employment', 'RemoteWork', 'LearnCodeChoose']

for col in categorical_cols:
    if col in df_2.columns:
        print(f"\n{'='*40}")
        print(f"Exact unique values for: {col}")
        print(f"{'='*40}")
        # tolist() ensures the output isn't truncated in your notebook
        print(df_2[col].value_counts(dropna=False))


# 2. Check the exact individual items for semicolon-separated columns
# Doing value_counts on these directly will print thousands of combinations (e.g. "MySQL;PostgreSQL")
# This explodes the lists so you see the exact names of the individual databases.
list_cols = ['DatabaseHaveWorkedWith', 'DatabaseAdmired']

for col in list_cols:
    if col in df_2.columns:
        print(f"\n{'='*40}")
        print(f"Exact individual values for: {col}")
        print(f"{'='*40}")
        
        # Split, explode, clean whitespace, and get unique items
        unique_items = set(df_2[col].astype(str).str.split(';').explode().str.strip())
        
        # Sort alphabetically for easy reading
        print(sorted(list(unique_items)))


Exact unique values for: Employment
Employment
Employed                                                18921
Independent contractor, freelancer, or self-employed     3507
Not employed                                              680
Student                                                   673
I prefer not to say                                        91
Retired                                                    89
Name: count, dtype: int64

Exact unique values for: RemoteWork
RemoteWork
Remote                                                                          7112
Hybrid (some remote, leans heavy to in-person)                                  4058
Hybrid (some in-person, leans heavy to flexibility)                             3641
Not Specified                                                                   3484
In-person                                                                       3141
Your choice (very flexible, you can come in when you want or just as needed)    2

In [ ]:
# --- 1. LearnCodeChoose ---
# Encoding: "Yes, I am new to coding" -> 1, All else -> 0
df_2['LearnCodeChoose_Encoded'] = df_2['LearnCodeChoose'].apply(
    lambda x: 1 if "Yes, I am new to coding" in str(x) else 0
)

# --- 2. Employment ---
# Removing "Not employed"
df_2 = df_2[df_2['Employment'] != 'Not employed']


emp_map = {
    'Independent contractor, freelancer, or self-employed': 1,
    'Retired': 1,
    'Employed': 2,
    'I prefer not to say': 3,
    'Student': 3,
}
df_2['Employment_Encoded'] = df_2['Employment'].map(emp_map).fillna(3) # Defaulting any leftovers to 3

# --- 3. RemoteWork ---
# Encoding: Remote -> 1, Hybrid -> 2, In-person -> 3
remote_map = {
    'Remote': 1,
    'Your choice (very flexible, you can come in when you want or just as needed)': 1,
    'Hybrid (some remote, leans heavy to in-person)': 2,
    'Hybrid (some in-person, leans heavy to flexibility)': 2,
    'Not Specified': 2, # Grouping 'Not Specified' with Hybrid as it's the 'neutral' middle
    'In-person': 3
}
df_2['RemoteWork_Encoded'] = df_2['RemoteWork'].map(remote_map)

# --- 4. DatabaseHaveWorkedWith ---
# Encoding: Datomic/CockroachDB -> 1, All else -> 0
elite_worked = ['Datomic', 'Cockroachdb']
df_2['DB_Worked_Elite'] = df_2['DatabaseHaveWorkedWith'].apply(
    lambda x: 1 if any(db in str(x) for db in elite_worked) else 0
)

# --- 5. DatabaseAdmired ---
# Encoding: Cassandra/Cosmos DB -> 1, All else -> 0
elite_admired = ['Cassandra', 'Cosmos DB']
df_2['DB_Admired_Elite'] = df_2['DatabaseAdmired'].apply(
    lambda x: 1 if any(db in str(x) for db in elite_admired) else 0
)

In [8]:
# List of raw columns to drop
raw_cols = ['RemoteWork', 'Employment', 'LearnCodeChoose', 'DatabaseHaveWorkedWith', 'DatabaseAdmired']

# Keep only the encoded versions and your targets
df_final_numeric = df_2.drop(columns=raw_cols)

print("Pre-processing complete. Final columns for modeling:")
print(df_final_numeric.columns.tolist())

Pre-processing complete. Final columns for modeling:
['JobSatLevel', 'JobSat_Score', 'LearnCodeChoose_Encoded', 'Employment_Encoded', 'RemoteWork_Encoded', 'DB_Worked_Elite', 'DB_Admired_Elite']


# For cleaning pipeline code

In [ ]:
def preprocess_survey_data(df, is_train=True):
    """
    Cleans and encodes survey data for machine learning.
    
    Parameters:
    df (DataFrame): The raw pandas dataframe.
    is_train (bool): Set to True if processing training data (handles target columns).
                     Set to False if processing unseen test data.
    """
    # 1. Feature Selection
    # If it's test data, the TA might not give you the JobSatLevel column!
    if is_train:
        df = df.dropna(subset=['JobSatLevel']).copy()
        sat_mapping = {'Low': 0, 'Average': 1, 'High': 2}
        df['JobSat_Score'] = df['JobSatLevel'].map(sat_mapping)
        selected_features = ['JobSatLevel', 'JobSat_Score', 'RemoteWork', 'Employment', 
                             'LearnCodeChoose', 'DatabaseHaveWorkedWith', 'DatabaseAdmired']
    else:
        # Test data doesn't have the target
        selected_features = ['RemoteWork', 'Employment', 'LearnCodeChoose', 
                             'DatabaseHaveWorkedWith', 'DatabaseAdmired']
    
    # Check if all selected columns actually exist in the dataframe to prevent errors
    existing_features = [col for col in selected_features if col in df.columns]
    df_clean = df[existing_features].copy()

    # 2. Fill Nulls
    if 'LearnCodeChoose' in df_clean:
        df_clean['LearnCodeChoose'] = df_clean['LearnCodeChoose'].fillna('No, I am not new to coding')
    if 'RemoteWork' in df_clean:
        df_clean['RemoteWork'] = df_clean['RemoteWork'].fillna('Not Specified')
    if 'DatabaseAdmired' in df_clean:
        df_clean['DatabaseAdmired'] = df_clean['DatabaseAdmired'].fillna('None')
    if 'DatabaseHaveWorkedWith' in df_clean:
        df_clean['DatabaseHaveWorkedWith'] = df_clean['DatabaseHaveWorkedWith'].fillna('None')

    # 3. Encoding
    # --- LearnCodeChoose ---
    if 'LearnCodeChoose' in df_clean:
        df_clean['LearnCodeChoose_Encoded'] = df_clean['LearnCodeChoose'].apply(
            lambda x: 1 if "Yes, I am new to coding" in str(x) else 0
        )

    # --- Employment ---
    # We map 'Not employed' to 3 instead of dropping rows, ensuring test data sizes match!
    emp_map = {
        'Independent contractor, freelancer, or self-employed': 1,
        'Retired': 1,
        'Employed': 2,
        'Employed': 2,
        'I prefer not to say': 3,
        'Student': 3,
        'Not employed': 4,
    }
    if 'Employment' in df_clean:
        df_clean['Employment_Encoded'] = df_clean['Employment'].map(emp_map).fillna(3)

    # --- RemoteWork ---
    remote_map = {
    'Remote': 1,
    'Your choice (very flexible, you can come in when you want or just as needed)': 1,
    'Hybrid (some remote, leans heavy to in-person)': 2,
    'Hybrid (some in-person, leans heavy to flexibility)': 2,
    'Not Specified': 2,
    'In-person': 3
    }
    if 'RemoteWork' in df_clean:
        df_clean['RemoteWork_Encoded'] = df_clean['RemoteWork'].map(remote_map).fillna(2) # Default to neutral

    # --- Databases ---
    elite_worked = ['Datomic', 'Cockroachdb']
    elite_admired = ['Cassandra', 'Cosmos DB']
    
    if 'DatabaseHaveWorkedWith' in df_clean:
        df_clean['DB_Worked_Elite'] = df_clean['DatabaseHaveWorkedWith'].apply(
            lambda x: 1 if any(db in str(x) for db in elite_worked) else 0
        )
        
    if 'DatabaseAdmired' in df_clean:
        df_clean['DB_Admired_Elite'] = df_clean['DatabaseAdmired'].apply(
            lambda x: 1 if any(db in str(x) for db in elite_admired) else 0
        )

    # 4. Drop Raw Columns
    raw_cols = ['RemoteWork', 'Employment', 'LearnCodeChoose', 'DatabaseHaveWorkedWith', 'DatabaseAdmired']
    cols_to_drop = [col for col in raw_cols if col in df_clean.columns]
    
    df_final_numeric = df_clean.drop(columns=cols_to_drop)
    
    return df_final_numeric